# Revision analyses (Scientific Reports, round 3)

Post-hoc analyses requested in the third review round. **CPU only, no LLM,
no GPU, nothing re-run.** It reads the files notebooks 02 and 03 already left
on your data root (cached LLM probabilities and abstract caches) and adds one
SBERT encoding pass, cached next to them.

| Part | Question (reviewer point) | Output CSV |
|---|---|---|
| 0 | Reproduction gate: do we recover the paper's (v6) numbers exactly? | `repro_gate.csv` |
| A1 | Is the LLM's increment over retrieval-only real? (R1.3) | `llm_increment.csv`, `llm_increment_per_goal.csv` |
| A2 | Interventional retrieval controls: shuffled pool labels, random neighbours, similarity bands (R1.5) | `retrieval_controls.csv`, `retrieval_controls_seeds.csv` |
| A3 | Labelled-pool size curve: where does the LLM matter? (R1.1, R1.3) | `pool_size_curve.csv`, `pool_size_curve_seeds.csv` |
| A4 | LLM increment by stratum: title-only, neighbour reliability, LLM uncertainty (R1.3) | `llm_increment_strata.csv` |
| B1 | Supervised baselines trained on the same pool labels (R1 "simpler alternatives") | `supervised_pool_baselines.csv` |

Output goes to a new folder, `<root>/aurora_sdg_graph_full/cross_model_comparison_v7/`.
Nothing written by earlier notebooks is read-modified or overwritten.

**Where the inputs are found.** The root is resolved as in notebooks 01-03
($SDG_ROOT, else config.yaml, else Drive on Colab, else ./sdg_data). If the
inputs are not under that root, the notebook searches your Drive (Colab) or
the parent folders (local) for a folder holding `aurora_sdg_graph_full/`,
prints what it found, and uses it. To skip the search, set `SDG_ROOT` in a
cell above this one. The repository's `artifacts/` folder is optional: it is
used when present, and otherwise not needed.

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────────
import os, sys

def _resolve_sdg_root():
    r = os.environ.get('SDG_ROOT')
    if r:
        return os.path.abspath(os.path.expanduser(r))
    for cand in ('config.yaml', os.path.join('..', 'config.yaml')):
        if os.path.exists(cand):
            with open(cand) as fh:
                for line in fh:
                    line = line.split('#')[0].strip()
                    if line.startswith('root:'):
                        v = line.split(':', 1)[1].strip().strip('"\'')
                        if v:
                            return os.path.abspath(os.path.expanduser(v))
    if 'google.colab' in sys.modules or os.path.isdir('/content'):
        try:
            from google.colab import drive
            if not os.path.exists('/content/drive/MyDrive'):
                drive.mount('/content/drive')
            return '/content/drive/MyDrive/sdg-llm-graph'
        except Exception:
            pass
    return os.path.abspath('./sdg_data')

RUN_DIR  = 'aurora_sdg_graph_full'                      # layout of notebooks 02/03
ABS_REL  = os.path.join(RUN_DIR, 'data_cache', 'abstracts_full.parquet')

def _search(top, max_depth=5):
    """Folders under `top` (depth-limited) holding a notebook-02/03 run
    (<dir>/aurora_sdg_graph_full/data_cache/abstracts_full.parquet), and
    released artifacts folders (<dir>/splits/test_labels.npz)."""
    roots, arts = [], []
    top = os.path.abspath(top)
    base = top.rstrip(os.sep).count(os.sep)
    for dirpath, dirnames, _ in os.walk(top):
        if dirpath.count(os.sep) - base >= max_depth:
            dirnames[:] = []
        dirnames[:] = [d for d in dirnames if not d.startswith('.')
                       and d not in ('node_modules', '__pycache__')]
        if RUN_DIR in dirnames and os.path.exists(os.path.join(dirpath, ABS_REL)):
            roots.append(dirpath)
        if 'splits' in dirnames and os.path.exists(
                os.path.join(dirpath, 'splits', 'test_labels.npz')):
            arts.append(dirpath)
    return roots, arts

SDG_ROOT = _resolve_sdg_root()                 # mounts Drive on Colab
ON_COLAB = os.path.isdir('/content/drive/MyDrive')   # so test only after mounting
_explicit = bool(os.environ.get('SDG_ROOT'))

def _artifacts_near(root):
    cands = [os.environ.get('SDG_ARTIFACTS'), os.path.join(root, 'artifacts'),
             os.path.join('..', 'artifacts'), 'artifacts']
    for c in cands:
        if c and os.path.exists(os.path.join(c, 'splits', 'test_labels.npz')):
            return os.path.abspath(c)
    return None

ARTIFACTS = _artifacts_near(SDG_ROOT)

if not os.path.exists(os.path.join(SDG_ROOT, ABS_REL)) and ARTIFACTS is None:
    if _explicit:
        raise FileNotFoundError(
            f'SDG_ROOT={SDG_ROOT} holds no {ABS_REL} and no artifacts/ folder was '
            'found. Point SDG_ROOT at the folder that contains aurora_sdg_graph_full/.')
    top = '/content/drive/MyDrive' if ON_COLAB else os.path.abspath('..')
    print(f'Inputs not under {SDG_ROOT}; searching {top} (this can take a minute on Drive)...')
    roots, arts = _search(top)
    for r in roots: print(f'  found run data : {r}')
    for a in arts:  print(f'  found artifacts: {a}')
    # prefer a root whose run also has the cached predictions of notebook 02
    def _has_preds(r):
        return os.path.isdir(os.path.join(r, RUN_DIR, 'results_v5_multi_gemma4-26b_n10000'))
    ranked = sorted(roots, key=lambda r: (not _has_preds(r), len(r)))
    if len(ranked) > 1 and _has_preds(ranked[0]) == _has_preds(ranked[1]):
        raise RuntimeError('Several candidate data roots were found (listed above). '
                           'Set SDG_ROOT to the one you want, in a cell above this one:\n'
                           "  import os; os.environ['SDG_ROOT'] = '<folder>'")
    if ranked:
        SDG_ROOT = ranked[0]
    elif arts:
        ARTIFACTS = arts[0]
        SDG_ROOT = os.path.dirname(arts[0])
    else:
        raise FileNotFoundError(
            f'No run data ({ABS_REL}) and no artifacts/ folder found under {top}. '
            "Set SDG_ROOT to the folder holding aurora_sdg_graph_full/, or "
            'SDG_ARTIFACTS to the repository\'s artifacts/ folder.')
    ARTIFACTS = ARTIFACTS or _artifacts_near(SDG_ROOT)

DRIVE_ROOT = os.path.join(SDG_ROOT, RUN_DIR)
DATA_CACHE = os.path.join(DRIVE_ROOT, 'data_cache')
OUT        = os.path.join(DRIVE_ROOT, 'cross_model_comparison_v7')
os.makedirs(DATA_CACHE, exist_ok=True)
os.makedirs(OUT, exist_ok=True)

# Input folders, in order of preference: the run's own cache, then the release.
_cache_dirs = [DATA_CACHE] + ([os.path.join(ARTIFACTS, 'data_cache')] if ARTIFACTS else [])

def _locate(fname, subdirs):
    """First existing <dir>/<fname> among the given directories, else None."""
    for d in subdirs:
        p = os.path.join(d, fname)
        if os.path.exists(p):
            return p
    return None

print(f'SDG_ROOT   : {SDG_ROOT}')
print(f'DATA_CACHE : {DATA_CACHE}')
print(f'ARTIFACTS  : {ARTIFACTS or "(not found - not needed)"}')
print(f'OUT        : {OUT}')

In [ ]:
import importlib.util, subprocess
for _pkg, _mod in [('pyarrow', 'pyarrow'), ('scikit-learn', 'sklearn'),
                   ('scipy', 'scipy'), ('pandas', 'pandas')]:
    if importlib.util.find_spec(_mod) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _pkg], check=True)

import json, hashlib, time, re
import numpy as np, pandas as pd
from scipy import stats

# ---- CONFIG -----------------------------------------------------------------
SEED       = 42
MODELS     = ['gemma4-26b', 'mistral-24b', 'qwen3-32b', 'mixtral-8x7b']
PRIMARY    = 'gemma4-26b'
K_HEAD     = 50
CANON_GRID = np.arange(0.10, 0.851, 0.05)     # paper's threshold grid (16 points)
LAM_GRID   = np.arange(0.0, 1.01, 0.1)        # identical to notebook 03, Cell U
N_BOOT     = 1000
BOOT_SEED  = 12345                            # identical to notebook 03, Cell U
SBERT_NAME = 'all-MiniLM-L6-v2'

N_CONTROL_SEEDS = 20                          # A2: shuffled labels, random neighbours
SIM_BANDS  = [(0, 50), (50, 100), (100, 200), (200, 400), (400, 800),
              (1000, 1050), ('bottom', 50)]   # A2: neighbour rank bands
POOL_SIZES = [50, 100, 250, 500, 1000, 2000]  # A3 (full pool 3054 added automatically)
N_POOL_SEEDS = 10
LOGREG_C   = [0.1, 1.0, 10.0]                 # B1: selected on validation

# Split fingerprints (SHA-1 of the ordered DOI list, first 12 hex), as in the paper.
FINGERPRINT = {'val': 'b212c22b4951', 'test': '4c5581303537', 'pool': 'c8ffa1517ecc'}

# The paper's numbers (cross_model_comparison_v6, written by notebook 03 Cell U).
# The gate below must reproduce them; if it fails, results are not comparable.
V6_REF = {
    'gemma4-26b':   {'B1': 0.5216177784976842, 'B2': 0.5286151349954628,
                     'doc_neighbours': 0.611613262299146,  'lambda': 0.7},
    'mistral-24b':  {'B1': 0.4787128317504444, 'B2': 0.4589641221985147,
                     'doc_neighbours': 0.5940953946602423, 'lambda': 0.8},
    'qwen3-32b':    {'B1': 0.4240813821399711, 'B2': 0.4024134188787943,
                     'doc_neighbours': 0.5831047725927838, 'lambda': 0.7},
    'mixtral-8x7b': {'B1': 0.2198128864316267, 'B2': 0.1829298028348064,
                     'doc_neighbours': 0.5821240490870296, 'lambda': 1.0},
}
V6_RETRIEVAL_ONLY = 0.5821240490870296
V6_BOOT_NBR_B2    = 0.0829776942864933        # gemma, doc_neighbours - B2, mean

STRICT_REPRO_GATE = os.environ.get('SDG_STRICT_GATE', '1') != '0'
REPRO_TOL = 1e-6

## 1. Splits, labels, texts, predictions
Same sources and slicing as notebook 03: the abstract cache sliced
sequentially (few-shot 40, val 200, test 10,000) and the neighbour pool cache
filtered at 50 characters. Labels are the parquet's goal columns. Every split
is checked against its published fingerprint.

In [ ]:
def _fp(dois):
    return hashlib.sha1(''.join(dois).encode('utf-8')).hexdigest()[:12]

p_abs  = _locate('abstracts_full.parquet', _cache_dirs)
p_pool = _locate(f'neighbour_pool_abstracts_n5000_seed{SEED}.parquet', _cache_dirs)
assert p_abs,  f'abstracts_full.parquet not found in {_cache_dirs}'
assert p_pool, f'neighbour_pool_abstracts_n5000_seed{SEED}.parquet not found in {_cache_dirs}'
print(f'abstracts : {p_abs}\npool      : {p_pool}')

df_all  = pd.read_parquet(p_abs)
df_val  = df_all.iloc[40:240].reset_index(drop=True)
df_test = df_all.iloc[240:10240].reset_index(drop=True)
df_pool = pd.read_parquet(p_pool)
df_pool = df_pool[df_pool['abstract'].fillna('').str.len() > 50].reset_index(drop=True)

for name, df in [('val', df_val), ('test', df_test), ('pool', df_pool)]:
    fp = _fp(df['doi'].astype(str).tolist())
    assert fp == FINGERPRINT[name], \
        f'{name}: fingerprint {fp} != published {FINGERPRINT[name]} (different cache?)'
    print(f'  {name:5s} n={len(df):5d}  fingerprint {fp}  OK')

goal_cols = sorted([c for c in df_all.columns if re.fullmatch(r'sdg_\d+', c)],
                   key=lambda c: int(c.split('_')[1]))
assert len(goal_cols) == 17
y_val  = df_val[goal_cols].values.astype(int)
y_test = df_test[goal_cols].values.astype(int)
y_pool = df_pool[goal_cols].values.astype(np.float32)

# Cross-check against the released label arrays when the artifacts are around.
if ARTIFACTS:
    for name, y in [('val', y_val), ('test', y_test), ('pool', y_pool)]:
        ref = np.load(os.path.join(ARTIFACTS, 'splits', f'{name}_labels.npz'))['goal']
        assert np.array_equal(y, ref), f'{name}: labels differ from artifacts/splits'
    print('  labels identical to artifacts/splits')

def _text(df):
    return (df['title'].fillna('') + '. ' + df['abstract'].fillna('')).tolist()

txt_val, txt_test, txt_pool = _text(df_val), _text(df_test), _text(df_pool)
has_abstract = (df_test['abstract'].fillna('').str.len() > 50).values

# Raw LLM predictions: notebook 02's result folders, else the released .npz.
cached = {}
for m in MODELS:
    tag = f'v5_multi_{m}_n10000'
    rd = os.path.join(DRIVE_ROOT, f'results_{tag}')
    files = {f'{k}_{s}': os.path.join(rd, f'{k}_probs_{s}_{tag}.npy')
             for k in ('goal', 'target') for s in ('val', 'test')}
    npz = (_locate(f'raw_llm_predictions_{m}_n10000.npz',
                   [os.path.join(ARTIFACTS, 'predictions')]) if ARTIFACTS else None)
    if all(os.path.exists(f) for f in files.values()):
        cached[m] = {k: np.load(f) for k, f in files.items()}
        src = rd
    elif npz:
        z = np.load(npz)
        cached[m] = {f'{k}_{s}': z[f'{k}_probs_{s}_{tag}']
                     for k in ('goal', 'target') for s in ('val', 'test')}
        src = npz
    else:
        raise FileNotFoundError(f'{m}: no cached predictions in {rd} and no released .npz')
    assert cached[m]['goal_test'].shape == (10000, 17)
    print(f'  {m:13s} predictions from {src}')

## 2. SBERT embeddings (computed once, cached under DATA_CACHE)

In [ ]:
# Keyed by model name and the three split fingerprints. A copy shipped in
# artifacts/data_cache is used if present, so the neighbour vote does not depend
# on the locally installed sentence-transformers / torch versions.
_EMB_NAME = (f'sbert_{SBERT_NAME}_val{FINGERPRINT["val"]}'
             f'_test{FINGERPRINT["test"]}_pool{FINGERPRINT["pool"]}.npz')
EMB_CACHE = _locate(_EMB_NAME, _cache_dirs) or os.path.join(DATA_CACHE, _EMB_NAME)

if os.path.exists(EMB_CACHE):
    z = np.load(EMB_CACHE)
    val_emb, test_emb, pool_emb = z['val'], z['test'], z['pool']
    print(f'Embeddings loaded from cache: {EMB_CACHE}')
else:
    if importlib.util.find_spec('sentence_transformers') is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                        'sentence-transformers'], check=True)
    from sentence_transformers import SentenceTransformer
    sbert = SentenceTransformer(SBERT_NAME)
    t0 = time.time()
    enc = lambda t: sbert.encode(t, normalize_embeddings=True, batch_size=64,
                                 show_progress_bar=False).astype(np.float32)
    pool_emb, val_emb, test_emb = enc(txt_pool), enc(txt_val), enc(txt_test)
    np.savez_compressed(EMB_CACHE, val=val_emb, test=test_emb, pool=pool_emb)
    print(f'Encoded in {time.time()-t0:.0f}s, cached to {EMB_CACHE}')

print(f'pool {pool_emb.shape}  val {val_emb.shape}  test {test_emb.shape}')

## 3. Scoring, neighbour vote, mixing (verbatim from notebook 03, Cells U/R)

In [ ]:
TARGETS_PER_GOAL = [7,8,13,10,9,8,5,12,8,10,10,11,5,10,12,12,19]
GOAL_TO_TARGETS, _i = {}, 0
for g, n in enumerate(TARGETS_PER_GOAL, start=1):
    GOAL_TO_TARGETS[g] = list(range(_i, _i + n)); _i += n

def aggregate_targets_to_goals(target_probs):
    out = np.zeros((target_probs.shape[0], 17), dtype=np.float32)
    for g in range(1, 18):
        out[:, g-1] = target_probs[:, GOAL_TO_TARGETS[g]].max(axis=1)
    return out

def _f1_bin(yt, yp):
    tp = float(np.count_nonzero(yt & yp))
    fp = float(np.count_nonzero((~yt) & yp))
    fn = float(np.count_nonzero(yt & (~yp)))
    d = 2*tp + fp + fn
    return 0.0 if d == 0 else 2*tp/d

def fit_thresholds(probs, labels, grid=CANON_GRID):
    yb = labels.astype(bool)
    thr = np.zeros(probs.shape[1])
    for c in range(probs.shape[1]):
        col, yc = probs[:, c], yb[:, c]
        best_f, best_t = -1.0, float(grid[0])
        for t in grid:
            f = _f1_bin(yc, col >= t)
            if f > best_f:
                best_f, best_t = f, float(t)
        thr[c] = best_t
    return thr

def per_class_f1(probs, labels, thr):
    yb, pred = labels.astype(bool), probs >= thr
    return np.array([_f1_bin(yb[:, c], pred[:, c]) for c in range(probs.shape[1])])

def score(probs, labels, thr):
    return float(per_class_f1(probs, labels, thr).mean())

def fit_and_score(p_val, p_test):
    thr = fit_thresholds(p_val, y_val)
    return score(p_val, y_val, thr), score(p_test, y_test, thr), thr

def vote_from_idx(sims, idx, labels):
    """Eq. 5: similarity-weighted vote over the neighbour index set idx (n x K)."""
    out = np.zeros((idx.shape[0], labels.shape[1]), dtype=np.float64)
    for i in range(idx.shape[0]):
        nn = idx[i]
        w = np.clip(sims[i, nn], 0, None); Z = w.sum()
        out[i] = (w @ labels[nn]) / Z if Z > 0 else labels[nn].mean(0)
    return out

def nbr_vote(q_emb, p_emb, p_lab, K):
    """Top-K vote, exactly as Cell U's nbr_vote(weighted=True)."""
    sims = q_emb @ p_emb.T
    idx = np.argpartition(-sims, K-1, axis=1)[:, :K]
    return vote_from_idx(sims, idx, p_lab)

def calibrate_mix(gv, gt, ev, et):
    """Global lambda on validation (first maximum), as Cell U. Returns dict."""
    best = None
    for lam in LAM_GRID:
        v, t, thr = fit_and_score((1-lam)*gv + lam*ev, (1-lam)*gt + lam*et)
        if best is None or v > best['val']:
            best = {'lam': float(lam), 'val': v, 'test': t, 'thr': thr,
                    'p_test': (1-lam)*gt + lam*et}
    return best

def boot_diff(pA, thrA, pB, thrB, mask=None, n_boot=N_BOOT, seed=BOOT_SEED):
    """Paired document-level bootstrap of macro-F1(A) - macro-F1(B), Cell U."""
    yt = y_test if mask is None else y_test[mask]
    pA = pA if mask is None else pA[mask]
    pB = pB if mask is None else pB[mask]
    rs = np.random.RandomState(seed); n = pA.shape[0]; d = np.empty(n_boot)
    for k in range(n_boot):
        b = rs.randint(0, n, n)
        d[k] = score(pA[b], yt[b], thrA) - score(pB[b], yt[b], thrB)
    return float(d.mean()), float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))

def mcnemar(pA, thrA, pB, thrB):
    """Per-(paper, goal) correctness. b: A right, B wrong; c: the converse."""
    okA = (pA >= thrA).astype(int) == y_test
    okB = (pB >= thrB).astype(int) == y_test
    b, c = int((okA & ~okB).sum()), int((~okA & okB).sum())
    if b + c >= 25:
        p = float(stats.chi2.sf((abs(b - c) - 1) ** 2 / (b + c), 1))
    else:
        p = float(stats.binomtest(b, b + c, 0.5).pvalue) if b + c else 1.0
    return b, c, p

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

def ovr_logreg(Xtr, ytr, Xv, Xt, C):
    """One-vs-rest, class-balanced logistic regression; a goal with a single
    class in the training labels gets its constant training frequency."""
    pv = np.zeros((Xv.shape[0], 17)); pt = np.zeros((Xt.shape[0], 17))
    for g in range(17):
        yg = ytr[:, g].astype(int)
        if yg.min() == yg.max():
            pv[:, g] = pt[:, g] = float(yg[0]); continue
        clf = LogisticRegression(C=C, class_weight='balanced', max_iter=2000,
                                 random_state=SEED)
        clf.fit(Xtr, yg)
        pv[:, g] = clf.predict_proba(Xv)[:, 1]; pt[:, g] = clf.predict_proba(Xt)[:, 1]
    return pv, pt

def supervised(features, idx):
    """Train on pool rows idx; C chosen on validation (first maximum)."""
    if features == 'TF-IDF':
        vec = TfidfVectorizer(sublinear_tf=True, min_df=2, max_df=0.9,
                              ngram_range=(1, 2), stop_words='english',
                              max_features=50000)
        Xtr = vec.fit_transform([txt_pool[i] for i in idx])
        Xv, Xt = vec.transform(txt_val), vec.transform(txt_test)
    else:
        Xtr, Xv, Xt = pool_emb[idx], val_emb, test_emb
    best = None
    for C in LOGREG_C:
        pv, pt = ovr_logreg(Xtr, y_pool[idx], Xv, Xt, C)
        v, t, thr = fit_and_score(pv, pt)
        if best is None or v > best['val']:
            best = {'val': v, 'test': t, 'thr': thr, 'C': C, 'pv': pv, 'pt': pt}
    return best

# Shared quantities used by every part below.
ev_val  = nbr_vote(val_emb,  pool_emb, y_pool, K_HEAD)
ev_test = nbr_vote(test_emb, pool_emb, y_pool, K_HEAD)
SIMS_TEST = test_emb @ pool_emb.T
SIMS_VAL  = val_emb  @ pool_emb.T

BASE = {}
for m in MODELS:
    gv = cached[m]['goal_val'].astype(np.float64); gt = cached[m]['goal_test'].astype(np.float64)
    av = aggregate_targets_to_goals(cached[m]['target_val'])
    at = aggregate_targets_to_goals(cached[m]['target_test'])
    b1v, b1t, thr1 = fit_and_score(gv, gt)
    b2v, b2t, thr2 = fit_and_score(av, at)
    BASE[m] = dict(gv=gv, gt=gt, at=at, b1=b1t, b2=b2t, thr1=thr1, thr2=thr2,
                   mix=calibrate_mix(gv, gt, ev_val, ev_test))
rv, rt, thr_r = fit_and_score(ev_val, ev_test)
RETR = dict(val=rv, test=rt, thr=thr_r)

## 0. Reproduction gate
Every number below must match the v6 results that back the paper's tables.
The B1/B2 rows test the cached predictions and the scoring; the
doc_neighbours, lambda, retrieval-only and bootstrap rows also test the SBERT
embeddings. If only the latter fail, the embeddings differ from the original
run (library versions); see the README, section 4c.

In [ ]:
gate = []
def _chk(what, got, ref):
    gate.append({'check': what, 'got': got, 'v6': ref, 'abs_diff': abs(got - ref),
                 'ok': abs(got - ref) <= REPRO_TOL})
for m in MODELS:
    ref = V6_REF[m]
    _chk(f'{m} B1 test', BASE[m]['b1'], ref['B1'])
    _chk(f'{m} B2 test', BASE[m]['b2'], ref['B2'])
    _chk(f'{m} doc_neighbours test', BASE[m]['mix']['test'], ref['doc_neighbours'])
    _chk(f'{m} lambda', BASE[m]['mix']['lam'], ref['lambda'])
_chk('retrieval-only test', RETR['test'], V6_RETRIEVAL_ONLY)
mu, _, _ = boot_diff(BASE[PRIMARY]['mix']['p_test'], BASE[PRIMARY]['mix']['thr'],
                     BASE[PRIMARY]['at'], BASE[PRIMARY]['thr2'])
_chk(f'{PRIMARY} bootstrap nbr-B2 mean', mu, V6_BOOT_NBR_B2)

df_gate = pd.DataFrame(gate)
df_gate.to_csv(os.path.join(OUT, 'repro_gate.csv'), index=False)
print(df_gate.to_string(index=False))
if not df_gate.ok.all():
    _emb_only = df_gate[~df_gate.ok].check.str.contains('doc_neighbours|lambda|retrieval|bootstrap').all()
    msg = ('REPRODUCTION GATE FAILED: results below are not comparable with the paper.'
           + (' Only embedding-dependent rows differ: the SBERT encoding is not identical'
              ' to the original run. Set SDG_STRICT_GATE=0 to continue; the new analyses'
              ' are then internally consistent but differ slightly from the paper.'
              if _emb_only else ' Cached predictions or scoring differ - do not continue.'))
    if STRICT_REPRO_GATE:
        raise RuntimeError(msg)
    print('\n!! ' + msg + ' (continuing because SDG_STRICT_GATE=0)\n')
else:
    print('\nREPRODUCTION GATE PASSED')

## A1. The LLM's increment over retrieval-only (R1.3)
doc_neighbours (calibrated mixture) minus retrieval-only (lambda = 1),
paired bootstrap over documents plus McNemar on per-(paper, goal) decisions.

In [ ]:
rows, pg_rows = [], []
pc_r = per_class_f1(ev_test, y_test, RETR['thr'])
for m in MODELS:
    B, mx = BASE[m], BASE[m]['mix']
    mu, lo, hi = boot_diff(mx['p_test'], mx['thr'], ev_test, RETR['thr'])
    b, c, p = mcnemar(mx['p_test'], mx['thr'], ev_test, RETR['thr'])
    mu2, lo2, hi2 = boot_diff(ev_test, RETR['thr'], B['at'], B['thr2'])
    pc_m = per_class_f1(mx['p_test'], y_test, mx['thr'])
    rows.append({'model': m, 'lambda': mx['lam'], 'B2': B['b2'],
                 'retrieval_only': RETR['test'], 'doc_neighbours': mx['test'],
                 'llm_increment': mx['test'] - RETR['test'],
                 'llm_inc_ci_lo': lo, 'llm_inc_ci_hi': hi,
                 'mcnemar_b_mix_right': b, 'mcnemar_c_retr_right': c, 'mcnemar_p': p,
                 'goals_mix_better': int((pc_m > pc_r).sum()),
                 'goals_mix_worse': int((pc_m < pc_r).sum()),
                 'retrieval_minus_B2': mu2, 'retr_B2_ci_lo': lo2, 'retr_B2_ci_hi': hi2})
    for g in range(17):
        pg_rows.append({'model': m, 'goal': g + 1, 'retrieval_only': pc_r[g],
                        'doc_neighbours': pc_m[g], 'diff': pc_m[g] - pc_r[g]})
df_a1 = pd.DataFrame(rows)
df_a1.to_csv(os.path.join(OUT, 'llm_increment.csv'), index=False)
pd.DataFrame(pg_rows).to_csv(os.path.join(OUT, 'llm_increment_per_goal.csv'), index=False)
print(df_a1[['model', 'lambda', 'retrieval_only', 'doc_neighbours', 'llm_increment',
             'llm_inc_ci_lo', 'llm_inc_ci_hi', 'mcnemar_p', 'goals_mix_better']]
      .round(4).to_string(index=False))

## A2. Interventional retrieval controls (R1.5)
Each control changes one ingredient of the neighbour vote and re-calibrates
lambda on validation, exactly as for the real vote.
* **shuffled labels** — pool rows keep their embeddings, labels are permuted.
  Removes the text-label link while keeping the label marginals.
* **random neighbours** — K pool papers drawn at random per query.
  Removes similarity while keeping the real labels.
* **similarity bands** — neighbours taken from rank band [a, b) instead of
  the top K: a dose-response test on neighbour similarity.

In [ ]:
def _control_row(name, seed, ev_v, ev_t, extra=None):
    rv_, rt_, _ = fit_and_score(ev_v, ev_t)
    out = []
    for m in MODELS:
        B = BASE[m]
        mx = calibrate_mix(B['gv'], B['gt'], ev_v, ev_t)
        out.append({'control': name, 'seed': seed, 'model': m,
                    'retrieval_only': rt_, 'lambda': mx['lam'], 'mix': mx['test'],
                    'mix_minus_B1': mx['test'] - B['b1'],
                    'mix_minus_B2': mx['test'] - B['b2'], **(extra or {})})
    return out

ctrl = _control_row('true top-50', None, ev_val, ev_test,
                    {'mean_cos': float(np.partition(SIMS_TEST, -K_HEAD, 1)[:, -K_HEAD:].mean())})

for s in range(N_CONTROL_SEEDS):
    perm = np.random.RandomState(SEED + s).permutation(len(y_pool))
    ctrl += _control_row('shuffled labels', s,
                         nbr_vote(val_emb, pool_emb, y_pool[perm], K_HEAD),
                         nbr_vote(test_emb, pool_emb, y_pool[perm], K_HEAD),
                         {'mean_cos': ctrl[0]['mean_cos']})   # same neighbours as true top-50
    rs = np.random.RandomState(SEED + 1000 + s)
    rand_idx = lambda n: np.stack([rs.choice(len(y_pool), K_HEAD, replace=False)
                                   for _ in range(n)])
    iv, it = rand_idx(len(y_val)), rand_idx(len(y_test))
    ctrl += _control_row('random neighbours', s,
                         vote_from_idx(SIMS_VAL, iv, y_pool),
                         vote_from_idx(SIMS_TEST, it, y_pool),
                         {'mean_cos': float(np.take_along_axis(SIMS_TEST, it, 1).mean())})

ORD_VAL  = np.argsort(-SIMS_VAL, axis=1)
ORD_TEST = np.argsort(-SIMS_TEST, axis=1)
for band in SIM_BANDS:
    if band[0] == 'bottom':
        sl, name = slice(-band[1], None), f'ranks bottom {band[1]}'
    else:
        sl, name = slice(band[0], band[1]), f'ranks {band[0]+1}-{band[1]}'
    iv, it = ORD_VAL[:, sl], ORD_TEST[:, sl]
    ctrl += _control_row(name, None, vote_from_idx(SIMS_VAL, iv, y_pool),
                         vote_from_idx(SIMS_TEST, it, y_pool),
                         {'mean_cos': float(np.take_along_axis(SIMS_TEST, it, 1).mean())})

df_ctrl_seeds = pd.DataFrame(ctrl)
df_ctrl_seeds.to_csv(os.path.join(OUT, 'retrieval_controls_seeds.csv'), index=False)
df_ctrl = (df_ctrl_seeds.groupby(['control', 'model'], sort=False)
           .agg(n_seeds=('retrieval_only', 'size'),
                mean_cos=('mean_cos', 'mean'),
                retrieval_only=('retrieval_only', 'mean'),
                retrieval_only_sd=('retrieval_only', 'std'),
                mix=('mix', 'mean'), mix_sd=('mix', 'std'),
                mix_max=('mix', 'max'), lambda_mean=('lambda', 'mean'),
                mix_minus_B1=('mix_minus_B1', 'mean'),
                mix_minus_B2=('mix_minus_B2', 'mean'))
           .reset_index())
df_ctrl.to_csv(os.path.join(OUT, 'retrieval_controls.csv'), index=False)
print(df_ctrl[df_ctrl.model == PRIMARY].drop(columns='model').round(4).to_string(index=False))

## A3. Labelled-pool size curve (R1.1, R1.3)
Random sub-pools of the 3,054-paper pool. K = min(50, n). Retrieval-only and
the calibrated mixture are recomputed for every sub-pool, and so is the
TF-IDF logistic-regression baseline of B1 (alone and mixed with the LLM);
B1/B2 do not depend on the pool. Shows how much labelled in-domain data the
LLM-free methods need before they match the LLM, and how the LLM's increment
changes with pool size.

In [ ]:
ps_rows = []
N_POOL = len(y_pool)
for n in POOL_SIZES + [N_POOL]:
    seeds = range(N_POOL_SEEDS) if n < N_POOL else [None]
    for s in seeds:
        sub = (np.arange(N_POOL) if s is None else
               np.sort(np.random.RandomState(SEED + 10 * n + s).choice(N_POOL, n, replace=False)))
        K = min(K_HEAD, n)
        ev_v = nbr_vote(val_emb, pool_emb[sub], y_pool[sub], K)
        ev_t = nbr_vote(test_emb, pool_emb[sub], y_pool[sub], K)
        rv_, rt_, _ = fit_and_score(ev_v, ev_t)
        sup = supervised('TF-IDF', sub)
        for m in MODELS:
            B = BASE[m]
            mx = calibrate_mix(B['gv'], B['gt'], ev_v, ev_t)
            ms = calibrate_mix(B['gv'], B['gt'], sup['pv'], sup['pt'])
            ps_rows.append({'pool_size': n, 'seed': s, 'K': K, 'model': m,
                            'B1': B['b1'], 'B2': B['b2'], 'retrieval_only': rt_,
                            'lambda': mx['lam'], 'mix': mx['test'],
                            'llm_increment': mx['test'] - rt_,
                            'retrieval_gain_over_B2': mx['test'] - B['b2'],
                            'tfidf_logreg': sup['test'], 'tfidf_C': sup['C'],
                            'llm_tfidf_mix': ms['test'], 'llm_tfidf_lambda': ms['lam']})
    print(f'  pool size {n:5d} done')
df_ps_seeds = pd.DataFrame(ps_rows)
df_ps_seeds.to_csv(os.path.join(OUT, 'pool_size_curve_seeds.csv'), index=False)
df_ps = (df_ps_seeds.groupby(['model', 'pool_size'])
         .agg(n_seeds=('mix', 'size'), B1=('B1', 'first'), B2=('B2', 'first'),
              retrieval_only=('retrieval_only', 'mean'),
              retrieval_only_sd=('retrieval_only', 'std'),
              mix=('mix', 'mean'), mix_sd=('mix', 'std'),
              lambda_mean=('lambda', 'mean'),
              llm_increment=('llm_increment', 'mean'),
              llm_increment_sd=('llm_increment', 'std'),
              tfidf_logreg=('tfidf_logreg', 'mean'),
              tfidf_logreg_sd=('tfidf_logreg', 'std'),
              llm_tfidf_mix=('llm_tfidf_mix', 'mean'),
              llm_tfidf_mix_sd=('llm_tfidf_mix', 'std'))
         .reset_index())
df_ps.to_csv(os.path.join(OUT, 'pool_size_curve.csv'), index=False)
print(df_ps[df_ps.model == PRIMARY].drop(columns='model').round(4).to_string(index=False))

## A4. Where does the LLM add to retrieval? (R1.3)
Thresholds stay at their validation-selected values; macro-F1 is computed
inside each test stratum. Strata: abstract present vs title only (abstract
<= 50 characters, the pool's own filter); neighbour reliability and LLM
uncertainty median splits, defined exactly as Diagnostic 3 (notebook 03).

In [ ]:
rel = np.partition(SIMS_TEST, -K_HEAD, axis=1)[:, -K_HEAD:].mean(axis=1)
st_rows = []
for m in MODELS:
    B, mx = BASE[m], BASE[m]['mix']
    p = np.clip(B['gt'], 1e-9, 1 - 1e-9)
    ent = -(p*np.log2(p) + (1-p)*np.log2(1-p)).mean(axis=1)
    strata = [('all', np.ones(len(y_test), bool)),
              ('abstract present', has_abstract), ('title only', ~has_abstract),
              ('neighbours reliable', rel > np.median(rel)),
              ('neighbours unreliable', rel <= np.median(rel)),
              ('LLM certain', ent <= np.median(ent)),
              ('LLM uncertain', ent > np.median(ent))]
    for name, msk in strata:
        yt = y_test[msk]
        f = lambda pp, th: score(pp[msk], yt, th)
        mu, lo, hi = boot_diff(mx['p_test'], mx['thr'], ev_test, RETR['thr'], mask=msk)
        st_rows.append({'model': m, 'stratum': name, 'n': int(msk.sum()),
                        'B2': f(B['at'], B['thr2']),
                        'retrieval_only': f(ev_test, RETR['thr']),
                        'doc_neighbours': f(mx['p_test'], mx['thr']),
                        'llm_increment': mu, 'ci_lo': lo, 'ci_hi': hi})
df_st = pd.DataFrame(st_rows)
df_st.to_csv(os.path.join(OUT, 'llm_increment_strata.csv'), index=False)
print(df_st[df_st.model == PRIMARY].drop(columns='model').round(4).to_string(index=False))

## B1. Supervised baselines trained on the same pool labels
The paper's B3 is trained on 240 papers. A fair "simpler alternative" to
retrieval uses the same 3,054 labelled pool papers. One-vs-rest logistic
regression (class-balanced) on TF-IDF and on the SBERT embeddings; C picked on
validation; thresholds per class on validation as everywhere else. We also mix
each supervised model with the LLM goal probabilities under the same global
lambda calibration, to see whether the LLM adds on top of it.

In [ ]:
sup_rows = []
for feat in ('TF-IDF', 'SBERT'):
    b = supervised(feat, np.arange(len(y_pool)))
    v, t, thr, C, pv, pt = b['val'], b['test'], b['thr'], b['C'], b['pv'], b['pt']
    mu, lo, hi = boot_diff(pt, thr, ev_test, RETR['thr'])
    sup_rows.append({'model': '-', 'method': f'{feat} + LogReg (pool)', 'C': C,
                     'lambda': None, 'val_f1': v, 'test_f1': t,
                     'comparison': 'minus retrieval-only',
                     'diff': mu, 'ci_lo': lo, 'ci_hi': hi})
    for m in MODELS:
        B = BASE[m]
        mx = calibrate_mix(B['gv'], B['gt'], pv, pt)
        mu, lo, hi = boot_diff(mx['p_test'], mx['thr'], pt, thr)
        mu2, lo2, hi2 = boot_diff(mx['p_test'], mx['thr'],
                                  B['mix']['p_test'], B['mix']['thr'])
        base = {'model': m, 'method': f'LLM + {feat} LogReg mix', 'C': C,
                'lambda': mx['lam'], 'val_f1': mx['val'], 'test_f1': mx['test']}
        sup_rows.append({**base, 'comparison': f'minus {feat} LogReg alone',
                         'diff': mu, 'ci_lo': lo, 'ci_hi': hi})
        sup_rows.append({**base, 'comparison': 'minus doc_neighbours',
                         'diff': mu2, 'ci_lo': lo2, 'ci_hi': hi2})
    print(f'  {feat} done (C={C})')
df_sup = pd.DataFrame(sup_rows)
df_sup.to_csv(os.path.join(OUT, 'supervised_pool_baselines.csv'), index=False)
print(df_sup.round(4).to_string(index=False))

## Summary

In [ ]:
print('Files written to', OUT)
for f in sorted(os.listdir(OUT)):
    print('  ', f)